# Dự Án Phát Hiện Phiên Không An Toàn Trong Mô Hình Zero Trust Sử Dụng AI

Dự án này sử dụng dataset SIEM để huấn luyện mô hình AI phát hiện phiên người dùng không an toàn dựa trên hành vi truy cập.

## 1. Xác Định Cấu Trúc Dự Án

Trước tiên, hãy xem các file chính trong dự án.

In [ ]:
import os
print("Cấu trúc dự án:")
for item in os.listdir('.'):
    print(f"- {item}")

## 2. Cài Đặt Môi Trường Python

Môi trường ảo Python đã được tạo. Để kích hoạt:

```bash
# Trên Windows
.venv\Scripts\activate

# Trên Linux/Mac
source .venv/bin/activate
```

Phiên bản Python: 3.12.10

## 3. Cài Đặt Phụ Thuộc

Các gói cần thiết đã được cài đặt: pandas, scikit-learn, numpy, jupyter, matplotlib, seaborn.

Nếu cần cài thêm:

```bash
pip install pandas scikit-learn numpy matplotlib seaborn
```

## 4. Thiết Lập Biến Môi Trường

Không cần thiết lập biến môi trường bổ sung cho dự án này.

## 5. Chạy Dự Án

### Tải và Xử Lý Dữ Liệu

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Tải dataset
df = pd.read_csv('advanced_siem_dataset.csv')
print(f"Dataset shape: {df.shape}")
print(df.head())

In [ ]:
# Xử lý dữ liệu thiếu
df = df.fillna("unknown")

# Chuyển đổi cột số
numeric_cols = ['duration', 'bytes', 'src_port', 'dst_port']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# Tạo nhãn
unsafe_count = (
    (df["severity"].isin(["high", "critical"])).astype(int) +
    (df["alert_type"].notna()).astype(int) +
    (df["duration"] > 5000).astype(int) +
    (df["bytes"] > 1000000).astype(int)
)
df["label"] = (unsafe_count >= 2).astype(int)

print("Phân phối nhãn:")
print(df["label"].value_counts())

In [ ]:
# Mã hóa cột phân loại
le = LabelEncoder()
categorical_cols = df.select_dtypes(include=["object", "string"]).columns
for col in categorical_cols:
    df[col] = df[col].astype(str)
    df[col] = le.fit_transform(df[col])

# Chuẩn hóa
scaler = StandardScaler()
X = df.drop("label", axis=1)
X_scaled = scaler.fit_transform(X)
y = df["label"]

# Chia tập train/test
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Huấn luyện mô hình
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Dự đoán
y_pred = model.predict(X_test)

# Đánh giá
accuracy = accuracy_score(y_test, y_pred)
print(f"Độ chính xác: {accuracy:.4f}")
print(classification_report(y_test, y_pred))

## 6. Kiểm Tra Hoạt Động và Gỡ Lỗi

### Trực Quan Hóa Kết Quả

In [ ]:
# Phân phối nhãn
plt.figure(figsize=(6,4))
sns.countplot(x='label', data=df)
plt.title('Phân Phối Nhãn (0: An Toàn, 1: Không An Toàn)')
plt.show()

# Ma trận nhầm lẫn
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Ma Trận Nhầm Lẫn')
plt.xlabel('Dự Đoán')
plt.ylabel('Thực Tế')
plt.show()

# Tầm quan trọng của features
feature_importance = model.feature_importances_
features = df.drop("label", axis=1).columns
plt.figure(figsize=(10,6))
sns.barplot(x=feature_importance, y=features)
plt.title('Tầm Quan Trọng Của Các Features')
plt.show()